# Day 8: Image Captioning + OCR

**Goal**: Extract text context (BLIP captions + EasyOCR) for all images and append to `images.csv`.

**Done checkpoint**:
- `images.csv` has `caption` and `ocr_text` columns populated
- Inspect 20 rows to verify caption quality

**Runtime**: GPU (T4). ~44k images with BLIP takes ~2-3 hours.  
Run with Colab Pro or leave it running overnight. Progress is saved every 200 images.

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/PinterestDemo'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
!pip install -q torch torchvision transformers Pillow pandas numpy tqdm pyyaml easyocr
print('Done.')

## 2. Verify GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
print(f'CUDA: {torch.cuda.is_available()}')

## 3. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE = Path(PROJECT_DIR)
config['paths']['metadata_csv'] = str(BASE / 'data/metadata/images.csv')
config['model']['device']       = 'cuda'
config['model']['blip_model']   = 'Salesforce/blip-image-captioning-base'

import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])
print(f'Dataset: {len(df):,} images')
print(f'Current columns: {list(df.columns)}')

## 4. Preview BLIP Captioning (Single Image Test)

In [ ]:
from src.models.captioner import ImageCaptioner
from PIL import Image
from IPython.display import display

captioner = ImageCaptioner(model_name=config['model']['blip_model'], device='cuda')

# Test on 5 random images
samples = df.sample(5)
for _, row in samples.iterrows():
    path = row['path']
    caption = captioner.caption_image(path)
    print(f'ID: {row["image_id"]} | Category: {row.get("category","")} | Color: {row.get("color","")}')
    print(f'  Caption: {caption}')
    display(Image.open(path).resize((150, 150)))
    print()

## 5. Preview EasyOCR (Single Image Test)

In [ ]:
from src.models.ocr import OCRExtractor

ocr = OCRExtractor(gpu=True)

# Test on 3 images (fashion images usually have little text, but some have brand names)
for _, row in df.sample(3).iterrows():
    text = ocr.extract_text(row['path'])
    print(f'ID: {row["image_id"]} | OCR text: "{text}"')

## 6. Run Full Batch Enrichment

In [ ]:
# Check how many images still need captioning
import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])

if 'caption' in df.columns:
    done = (df['caption'].notna()) & (df['caption'] != '')
    print(f'Already captioned: {done.sum():,} / {len(df):,}')
    print(f'Remaining:         {(~done).sum():,}')
else:
    print(f'No captions yet. Will caption all {len(df):,} images.')

In [ ]:
# Run enrichment — saves progress every 200 images
# Resume-safe: already-captioned images are skipped
from scripts.enrich_captions import enrich_captions
enrich_captions(config, batch_size=16, save_every=200)

## 7. Inspect Results

In [ ]:
import pandas as pd
from PIL import Image
from IPython.display import display

df = pd.read_csv(config['paths']['metadata_csv'])
captioned = df[df['caption'].notna() & (df['caption'] != '')]

print(f'Total images:    {len(df):,}')
print(f'With captions:   {len(captioned):,}')

has_ocr = df['ocr_text'].notna() & (df['ocr_text'] != '')
print(f'With OCR text:   {has_ocr.sum():,}')

print('\nSample captions:')
for _, row in captioned.sample(min(5, len(captioned))).iterrows():
    print(f'  [{row["image_id"]}] {row.get("category","")} | Caption: {row["caption"]}')
    if row.get('ocr_text'):
        print(f'    OCR: {row["ocr_text"]}')

## ✅ Day 8 Done Checkpoint

In [ ]:
import pandas as pd

df = pd.read_csv(config['paths']['metadata_csv'])
assert 'caption' in df.columns, 'caption column missing'
assert 'ocr_text' in df.columns, 'ocr_text column missing'

captioned_pct = (df['caption'].notna() & (df['caption'] != '')).mean() * 100

print('Day 8 COMPLETE')
print(f'  Captioned: {captioned_pct:.1f}% of images')
print(f'  Sample: {df[df["caption"].notna()]["caption"].iloc[0]}')